# RAGAS Evaluation Template (Colab)

Template นี้ใช้ประเมินไฟล์อินพุต RAGAS แบบประหยัดค่าใช้จ่าย (เริ่ม 5 ข้อ)

รูปแบบแต่ละแถวต้องมี `user_input`, `response`, `retrieved_contexts`, `reference` (เหมือน `ragas_input.json`)

- **ชุด eval 24 มี.ค. (49 แถว):** รัน `python evaluation/build_ragas_input.py` ในเครื่อง แล้วอัปโหลด `evaluation/eval_results/ragas_input_20260324.json`
- **ชุดเดิม:** อัปโหลด `ragas_input.json` ได้ตามเดิม

In [ ]:
!pip -q install ragas datasets langchain-openai langchain-community sentence-transformers python-dotenv

In [ ]:
from google.colab import files
uploaded = files.upload()  # อัปโหลด ragas_input_20260324.json หรือ ragas_input.json

In [ ]:
import os
from google.colab import userdata

# แนะนำให้ตั้ง Secret ใน Colab ชื่อ OPENROUTER_API_KEY
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
assert os.environ.get("OPENROUTER_API_KEY"), "Missing OPENROUTER_API_KEY"

In [ ]:
import json, math, warnings
from datetime import datetime
from datasets import Dataset
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms.base import LangchainLLMWrapper
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.run_config import RunConfig

warnings.filterwarnings("ignore", message=".*LangchainEmbeddingsWrapper is deprecated.*", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*LangchainLLMWrapper is deprecated.*", category=DeprecationWarning)

EVAL_MODEL = "anthropic/claude-3-haiku"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
EMBED_MODEL = "intfloat/multilingual-e5-large"
SAMPLE_SIZE = 5  # เปลี่ยนเป็น 0 เพื่อรันทั้งหมด
INPUT_FILE = "ragas_input_20260324.json"  # หรือ "ragas_input.json"

RUN_CONFIG = RunConfig(timeout=300, max_retries=8, max_workers=4)

llm = LangchainLLMWrapper(ChatOpenAI(
    model=EVAL_MODEL,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    timeout=RUN_CONFIG.timeout,
    max_tokens=2048,
    default_headers={"HTTP-Referer": "https://colab.research.google.com", "X-Title": "KKU-CS-Ragas-Colab"},
), run_config=RUN_CONFIG)

embeddings = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name=EMBED_MODEL), run_config=RUN_CONFIG)
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

data = json.load(open(INPUT_FILE, encoding="utf-8"))
if SAMPLE_SIZE > 0:
    data = data[:SAMPLE_SIZE]
dataset = Dataset.from_list(data)
print("Rows:", len(dataset), "Columns:", dataset.column_names)

In [ ]:
result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=llm,
    embeddings=embeddings,
    run_config=RUN_CONFIG,
    raise_exceptions=False,
    show_progress=True,
)

print("\n=== Ragas Summary ===")
for k, v in sorted(getattr(result, "_repr_dict", {}).items()):
    if isinstance(v, float) and math.isnan(v):
        print(f"{k}: nan")
    else:
        print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_file = f"ragas_result_{ts}.json"
result.to_pandas().to_json(out_file, orient="records", force_ascii=False, indent=2)
print("Saved:", out_file)

In [ ]:
from google.colab import files
files.download(out_file)